# Inferencia · Paso 1: segmentar en Colab (GPU)
**Reto CIMAT / BraTS 2018 — prototipo de investigación, no clínico.**

Flujo híbrido (D30):
1. **Aquí, en Colab con GPU:** nnU-Net segmenta al paciente con los 5 modelos de los folds, aplica el posprocesado de ET (D24) y guarda `segmentacion.nii.gz` en Drive. Tarda segundos por paciente.
2. **En tu computadora:** la radiómica y el pronóstico, con el entorno validado:
   ```
   python inferencia/predecir.py --caso CARPETA --edad 62 --segmentacion segmentacion.nii.gz
   ```

La radiómica no se corre en Colab a propósito: PyRadiomics necesita numpy 1.x y otras versiones podrían dar características distintas a las del modelo entrenado sin que nada avise.

### Antes de empezar
- *Entorno de ejecución → Cambiar tipo → GPU* (T4 alcanza).
- Los pesos ya están en Drive, en `MyDrive/reto_cimat/nnUNet_results/` (del entrenamiento).
- Sube cada paciente a **`MyDrive/reto_cimat/casos/<nombre>/`** con sus 4 resonancias: `t1`, `t1ce`, `t2` y `flair` en NIfTI, con nombre simple (`t1.nii.gz`) o de BraTS (`<id>_t1.nii.gz`). Deben estar en **formato BraTS**: sin cráneo, co-registradas, voxel de 1 mm y 240 × 240 × 155.

## 0. Parámetros

In [ ]:
CASOS = ['mi_paciente']      # carpetas dentro de MyDrive/reto_cimat/casos/
FOLDS = (0, 1, 2, 3, 4)      # los 5 modelos, promediados (como en predecir.py)
TTA = True                   # aumento con espejos: en GPU vale la pena
UMBRAL_ET = 500              # D24: resultados/segmentacion/postproceso_et.json, 'umbral_casos_nuevos'

DRIVE = '/content/drive/MyDrive/reto_cimat'
MODELO = f'{DRIVE}/nnUNet_results/Dataset501_BraTS2018/nnUNetTrainer_100epochs_ckpt5__nnUNetPlans__3d_fullres'

## 1. GPU, Drive y nnU-Net

In [ ]:
import os, json
import torch
assert torch.cuda.is_available(), 'No hay GPU: cambia el tipo de entorno'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
for f in ('plans.json', 'dataset.json') + tuple(f'fold_{k}/checkpoint_final.pth' for k in FOLDS):
    assert os.path.exists(f'{MODELO}/{f}'), f'falta {MODELO}/{f}'
print('pesos encontrados')

!pip install -q nnunetv2==2.8.1      # misma version que el entorno local 'inferencia'
import importlib.metadata as md
print('nnunetv2', md.version('nnunetv2'))

## 2. Registrar el entrenador
nnU-Net necesita encontrar la clase con la que se guardaron los pesos (la misma que en `entrenar_colab.ipynb`).

In [ ]:
import nnunetv2
destino = os.path.join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer', 'variants', 'nnUNetTrainer_100epochs_ckpt5.py')
open(destino, 'w').write(
    'from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs\n\n\n'
    'class nnUNetTrainer_100epochs_ckpt5(nnUNetTrainer_100epochs):\n'
    '    def on_train_start(self):\n'
    '        super().on_train_start()\n'
    '        self.save_every = 5\n')
print('entrenador registrado')

## 3. Funciones (las mismas reglas que `inferencia/predecir.py`)

In [ ]:
import glob, os, shutil
import numpy as np, nibabel as nib

MODALIDADES = ['t1', 't1ce', 't2', 'flair']          # orden de canales de nnU-Net: 0000..0003 (D23)
NNUNET_A_BRATS = {0: 0, 1: 2, 2: 1, 3: 4}            # nnU-Net 1 edema, 2 necrosis, 3 realce -> BraTS


def buscar_modalidades(carpeta):
    """t1.nii.gz o <id>_t1.nii.gz, sin confundir t1 con t1ce. Busca tambien un nivel mas abajo."""
    if not os.path.isdir(carpeta):
        base = os.path.dirname(carpeta)
        hay = sorted(os.listdir(base)) if os.path.isdir(base) else []
        raise FileNotFoundError(f'no existe {carpeta}. Carpetas en {base}: {hay or "ninguna"}. '
                                f'Pon en CASOS el nombre exacto de la carpeta del paciente.')
    archivos = glob.glob(f'{carpeta}/*.nii*') + glob.glob(f'{carpeta}/*/*.nii*')
    rutas = {}
    for mod in MODALIDADES:
        c = [f for f in archivos
             if os.path.basename(f).split('.')[0] == mod or os.path.basename(f).split('.')[0].endswith(f'_{mod}')]
        if len(c) != 1:
            vistos = [os.path.relpath(f, carpeta) for f in archivos] or os.listdir(carpeta)
            raise FileNotFoundError(f'en {carpeta} hay {len(c)} archivos para {mod!r} (se espera 1). '
                                    f'Archivos encontrados: {vistos}. Deben llamarse {mod}.nii.gz '
                                    f'o <id>_{mod}.nii.gz')
        rutas[mod] = c[0]
    return rutas


def avisos_formato(rutas):
    imgs = {m: nib.load(p) for m, p in rutas.items()}
    ref, avisos = imgs['t1'], []
    for m, im in imgs.items():
        if im.shape != ref.shape or not np.allclose(im.affine, ref.affine, atol=1e-3):
            avisos.append(f'{m}: forma o geometria distinta de t1 (no co-registradas)')
    if ref.shape != (240, 240, 155):
        avisos.append(f'tamano {ref.shape}; BraTS usa (240, 240, 155)')
    if not np.allclose(ref.header.get_zooms()[:3], 1, atol=1e-2):
        avisos.append(f'voxel {ref.header.get_zooms()[:3]} mm; BraTS usa 1 mm')
    if (np.asanyarray(ref.dataobj) == 0).mean() < 0.5:
        avisos.append('poco fondo en 0: probablemente conserva el craneo')
    return avisos


def posprocesar_a_brats(seg, umbral=UMBRAL_ET):
    """D24: ET predicho por debajo del umbral -> necrosis/no realzado; luego etiquetas BraTS 0/1/2/4."""
    seg = seg.astype(np.int16).copy()
    n_et = int((seg == 3).sum())
    descartado = 0 < n_et < umbral
    if descartado:
        seg[seg == 3] = 2
    salida = np.zeros_like(seg, dtype=np.uint8)
    for a, b in NNUNET_A_BRATS.items():
        salida[seg == a] = b
    return salida, n_et, descartado

## 4. Segmentar

In [ ]:
%%time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

pred = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=TTA,
                       device=torch.device('cuda'), verbose=False, allow_tqdm=True)
pred.initialize_from_trained_model_folder(MODELO, use_folds=FOLDS, checkpoint_name='checkpoint_final.pth')

for caso in CASOS:
    carpeta = f'{DRIVE}/casos/{caso}'
    rutas = buscar_modalidades(carpeta)
    for a in avisos_formato(rutas):
        print(f'AVISO {caso}: {a}')
    tmp = f'/content/tmp_{caso}'
    os.makedirs(tmp, exist_ok=True)
    pred.predict_from_files([[rutas[m] for m in MODALIDADES]], [f'{tmp}/nnunet'],
                            save_probabilities=False, overwrite=True,
                            num_processes_preprocessing=1, num_processes_segmentation_export=1)
    img = nib.load(f'{tmp}/nnunet.nii.gz')
    seg, n_et, descartado = posprocesar_a_brats(np.asanyarray(img.dataobj))
    final = nib.Nifti1Image(seg, img.affine, img.header)
    final.set_data_dtype(np.uint8)
    nib.save(final, f'{carpeta}/segmentacion.nii.gz')
    vol = {r: round(float(np.isin(seg, e).sum()) / 1000, 2) for r, e in
           {'WT': (1, 2, 4), 'TC': (1, 4), 'ET': (4,)}.items()}
    print(f'{caso}: {vol} cm3' + (f' | ET de {n_et} voxeles descartado (< {UMBRAL_ET})' if descartado else ''))
    shutil.rmtree(tmp)
print('\nmascaras guardadas en Drive: casos/<caso>/segmentacion.nii.gz')

## 5. Vista rápida (opcional)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
colores = ListedColormap([(0, 0, 0, 0), (0.9, 0.1, 0.1, 1), (0.1, 0.8, 0.2, 1), (1.0, 0.85, 0.0, 1)])
for caso in CASOS:
    carpeta = f'{DRIVE}/casos/{caso}'
    s = np.asanyarray(nib.load(f'{carpeta}/segmentacion.nii.gz').dataobj)
    s = np.select([s == 1, s == 2, s == 4], [1, 2, 3], 0)
    flair = np.asanyarray(nib.load(buscar_modalidades(carpeta)['flair']).dataobj)
    z = int(np.argmax((s > 0).sum(axis=(0, 1)))) if s.any() else s.shape[2] // 2
    plt.figure(figsize=(5, 5))
    plt.imshow(np.rot90(flair[:, :, z]), cmap='gray')
    plt.imshow(np.rot90(s[:, :, z]), cmap=colores, vmin=0, vmax=3, alpha=0.45, interpolation='nearest')
    plt.title(f'{caso} · corte {z} · rojo necrosis, verde edema, amarillo realce', fontsize=9)
    plt.axis('off'); plt.show()

## 6. Siguiente paso, en tu computadora
1. Descarga de Drive `casos/<caso>/segmentacion.nii.gz` (pesa unos 100 KB).
2. Con el entorno `inferencia` (o `radiomica`, porque con `--segmentacion` no hace falta PyTorch):
   ```
   python inferencia/predecir.py --caso CARPETA_CON_LAS_4_RESONANCIAS --edad 62 --segmentacion segmentacion.nii.gz
   ```
   Devuelve los volúmenes, la clase de supervivencia, los días estimados y la figura en `resultados_inferencia/<caso>/`.